# 序列逆置 （加注意力的seq2seq）
使用attentive sequence to sequence 模型将一个字符串序列逆置。例如 `OIMESIQFIQ` 逆置成 `QIFQISEMIO`(下图来自网络，是一个加attentino的sequence to sequence 模型示意图)
![attentive seq2seq](./seq2seq-attn.jpg)

In [1]:
import numpy as np
import tensorflow as tf
import collections
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras import layers, optimizers, datasets
import os,sys,tqdm

## 玩具序列数据生成
生成只包含[A-Z]的字符串，并且将encoder输入以及decoder输入以及decoder输出准备好（转成index）

In [2]:
import random
import string

def randomString(stringLength):
    """Generate a random string with the combination of lowercase and uppercase letters """

    letters = string.ascii_uppercase
    return ''.join(random.choice(letters) for i in range(stringLength))

def get_batch(batch_size, length):
    batched_examples = [randomString(length) for i in range(batch_size)]
    enc_x = [[ord(ch)-ord('A')+1 for ch in list(exp)] for exp in batched_examples]
    y = [[o for o in reversed(e_idx)] for e_idx in enc_x]
    dec_x = [[0]+e_idx[:-1] for e_idx in y]
    return (batched_examples, tf.constant(enc_x, dtype=tf.int32), 
            tf.constant(dec_x, dtype=tf.int32), tf.constant(y, dtype=tf.int32))
print(get_batch(2, 10))

(['PFILOSANOB', 'AQCOTXGYYP'], <tf.Tensor: shape=(2, 10), dtype=int32, numpy=
array([[16,  6,  9, 12, 15, 19,  1, 14, 15,  2],
       [ 1, 17,  3, 15, 20, 24,  7, 25, 25, 16]], dtype=int32)>, <tf.Tensor: shape=(2, 10), dtype=int32, numpy=
array([[ 0,  2, 15, 14,  1, 19, 15, 12,  9,  6],
       [ 0, 16, 25, 25,  7, 24, 20, 15,  3, 17]], dtype=int32)>, <tf.Tensor: shape=(2, 10), dtype=int32, numpy=
array([[ 2, 15, 14,  1, 19, 15, 12,  9,  6, 16],
       [16, 25, 25,  7, 24, 20, 15,  3, 17,  1]], dtype=int32)>)


# 建立sequence to sequence 模型

完成两空，模型搭建以及单步解码逻辑

In [6]:
class mySeq2SeqModel(keras.Model):
    def __init__(self):
        super(mySeq2SeqModel, self).__init__()
        self.v_sz = 27
        self.hidden = 128
        self.embed_layer = tf.keras.layers.Embedding(self.v_sz, 64)
        
        self.encoder_cell = tf.keras.layers.SimpleRNNCell(self.hidden)
        self.decoder_cell = tf.keras.layers.SimpleRNNCell(self.hidden)
        
        self.encoder = tf.keras.layers.RNN(
            self.encoder_cell, return_sequences=True, return_state=True
        )
        self.decoder = tf.keras.layers.RNN(
            self.decoder_cell, return_sequences=True, return_state=True
        )
        self.dense_attn = tf.keras.layers.Dense(self.hidden)
        self.dense = tf.keras.layers.Dense(self.v_sz)
        
    def call(self, enc_ids, dec_ids):
        '''
        完成带attention机制的 sequence2sequence 模型的搭建，模块已经在`__init__`函数中定义好，
        用双线性attention，或者自己改一下`__init__`函数做加性attention
        '''
        enc_emb = self.embed_layer(enc_ids)
        enc_out, enc_state = self.encoder(enc_emb)
        
        dec_emb = self.embed_layer(dec_ids)
        dec_out, _ = self.decoder(dec_emb, initial_state=enc_state)
        
        # 加性注意力: score = v^T tanh(W_h h_t + W_s s_i) 的简化实现
        enc_proj = self.dense_attn(enc_out)
        score = tf.reduce_sum(
            tf.nn.tanh(tf.expand_dims(dec_out, axis=2) + tf.expand_dims(enc_proj, axis=1)),
            axis=-1,
        )
        attn = tf.nn.softmax(score, axis=-1)
        context = tf.matmul(attn, enc_out)
        
        logits = self.dense(dec_out + context)
        return logits
    
    def encode(self, enc_ids):
        enc_emb = self.embed_layer(enc_ids)  # shape(b_sz, len, emb_sz)
        enc_out, enc_state = self.encoder(enc_emb)
        return enc_out, [enc_out[:, -1, :], enc_state]
    
    def get_next_token(self, x, state, enc_out):
        '''
        shape(x) = [b_sz,] 
        '''
        if isinstance(state, (list, tuple)) and len(state) == 2:
            _, dec_state = state
        else:
            dec_state = state
        
        inp_emb = self.embed_layer(x)
        h, new_state_list = self.decoder_cell(inp_emb, [dec_state])
        new_dec_state = new_state_list[0]
        
        enc_proj = self.dense_attn(enc_out)
        score = tf.reduce_sum(tf.nn.tanh(tf.expand_dims(h, 1) + enc_proj), axis=-1)
        attn = tf.nn.softmax(score, axis=-1)
        context = tf.squeeze(tf.matmul(tf.expand_dims(attn, 1), enc_out), axis=1)
        
        logits = self.dense(h + context)
        out = tf.argmax(logits, axis=-1, output_type=tf.int32)
        return out, [context, new_dec_state]

# Loss函数以及训练逻辑

In [4]:
def compute_loss(logits, labels):
    losses = tf.nn.sparse_softmax_cross_entropy_with_logits(
        logits=logits, labels=labels
    )
    return tf.reduce_mean(losses)

def train_one_step(model, optimizer, enc_x, dec_x, y):
    with tf.GradientTape() as tape:
        logits = model(enc_x, dec_x)
        loss = compute_loss(logits, y)

    grads = tape.gradient(loss, model.trainable_variables)
    grads_and_vars = [
        (g, v) for g, v in zip(grads, model.trainable_variables) if g is not None
    ]
    if not grads_and_vars:
        raise RuntimeError("梯度为空，请检查前向图是否可导。")
    optimizer.apply_gradients(grads_and_vars)
    return loss

def train(model, optimizer, seqlen, steps=1000):
    loss = 0.0
    for step in range(steps):
        _, enc_x, dec_x, y = get_batch(32, seqlen)
        loss = train_one_step(model, optimizer, enc_x, dec_x, y)
        if step % 200 == 0:
            print('step', step, ': loss', float(loss.numpy()))
    return loss

# 训练迭代

In [7]:
optimizer = optimizers.Adam(0.0005)
model = mySeq2SeqModel()
train(model, optimizer, seqlen=20)

step 0 : loss 3.3115005493164062
step 200 : loss 2.681567430496216
step 400 : loss 1.9356117248535156
step 600 : loss 1.4274795055389404
step 800 : loss 1.145120620727539


<tf.Tensor: shape=(), dtype=float32, numpy=0.9718313217163086>

# 测试模型逆置能力
首先要先对输入的一个字符串进行encode，然后在用decoder解码出逆置的字符串

测试阶段跟训练阶段的区别在于，在训练的时候decoder的输入是给定的，而在预测的时候我们需要一步步生成下一步的decoder的输入

In [8]:
def sequence_reversal():
    def decode(init_state, steps, enc_out):
        b_sz = tf.shape(init_state[0])[0]
        cur_token = tf.zeros(shape=[b_sz], dtype=tf.int32)
        state = init_state
        collect = []
        for i in range(steps):
            cur_token, state = model.get_next_token(cur_token, state, enc_out)
            collect.append(tf.expand_dims(cur_token, axis=-1))
        out = tf.concat(collect, axis=-1).numpy()
        out = [''.join([chr(idx+ord('A')-1) for idx in exp]) for exp in out]
        return out
    
    batched_examples, enc_x, _, _ = get_batch(32, 20)
    enc_out, state = model.encode(enc_x)
    return decode(state, enc_x.get_shape()[-1], enc_out), batched_examples

def is_reverse(seq, rev_seq):
    rev_seq_rev = ''.join([i for i in reversed(list(rev_seq))])
    if seq == rev_seq_rev:
        return True
    else:
        return False
print([is_reverse(*item) for item in list(zip(*sequence_reversal()))])
print(list(zip(*sequence_reversal())))

[False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False]
[('GCCGUZFGMHMKQQTSHKAM', 'NEHRBKMRZMEEGDLUGCCG'), ('TBZNGGMILAVUOGTOFHNK', 'BBLQBGXOUYJQILNPNWBC'), ('ASUTCFPZZPWAIZANNBHE', 'ETTSBZGGFPQBLLMJBUNA'), ('QRVQGAOMHNUCVTJAYGSZ', 'PLGVSFCETLRVMRAUQHRQ'), ('TIQUHJGTTBHIFGYIHYGG', 'XXOMGHXXRPTZTXEDOEYT'), ('QXLGWBISVSIFNBDIGSPL', 'ARAOUSWKNCPVMQPUXXIE'), ('YZCHAJBHDPJPIXNQTJOW', 'RAKLLXLCLYMHSBQAHCZY'), ('SJLWBWNXJPQLWPVWDVTQ', 'MOADMVPILLXEMNWBXLJS'), ('UDZUQSJOAJFCHQOXUNDH', 'OSJTOOGHQPJAOJSQUZCU'), ('IMEMPDQTZGMBZJLAITPM', 'ILXLUTQZQMMIUSEGXKMW'), ('DUUHFYHDPRCAIUCJJABN', 'BJVZADMUMHVABJYMOOBC'), ('RRPBNUJCOMRYNSSOMVGC', 'DRHLOSSBYRMUCJUNBPRR'), ('ITXGAUJYSMGHZCYCKXWW', 'DZWJLPZXHNMCYJUAGXTI'), ('HDVEBZRUQIGXVTAEHVGB', 'SGNTEJTRXGIZURZBEVDH'), ('OUYWMDQNZDPPSGUCWRGG', 'GURWWOWHPJJSFCDMWUUR'), ('OTCVDXLUYGBVTVTTQWYD',